Football Player Future G/A Predictor

Importing Modules

In [59]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

Reading the Data

In [15]:
df23 = pd.read_csv('data/football-data_23-24.csv') 
df24 = pd.read_csv('data/football-data_24-25.csv')
df25 = pd.read_csv('data/football-data_25-26.csv')

print(f"2023-2024 shape: {df23.shape}")
print(f"2024-2025 shape: {df24.shape}")
print(f"2025-2026 shape: {df25.shape}")

2023-2024 shape: (2852, 37)
2024-2025 shape: (2854, 267)
2025-2026 shape: (2779, 102)


Preparing and Cleaning the data

In [ ]:
base_features = ['Min', '90s', 'Gls', 'Ast', 'G+A', 'xG', 'npxG', 'xAG', 'PrgC', 'PrgP', 'PrgR']

df23_sub = df23[['Player', 'Age', 'Pos'] + base_features]
df23_sub = df23_sub.add_suffix('_23')
df23_sub = df23_sub.rename(columns={'Player_23': 'Player'})

df24_sub = df24[['Player'] + base_features]
df24_sub = df24_sub.add_suffix('_24')
df24_sub = df24_sub.rename(columns={'Player_24': 'Player'})

df25_target = df25[['Player', 'G+A']].rename(columns={'G+A': 'Target_G+A'})

cleandf = df23_sub.merge(df24_sub, on='Player', how='inner').merge(df25_target, on='Player', how='inner')
cleandf = cleandf.drop_duplicates(subset=['Player'])

print(f"Total players available for training: {cleandf.shape[0]}")
cleandf = cleandf.dropna(subset=['Age_23'])
print(f"Players left after dropping bad ages: {cleandf.shape[0]}")


Total players available for training: 1364
Players left after dropping bad ages: 1363


Training the model using the test data

In [60]:
features = [
    'Age_23', 'Min_23', '90s_23', 'G+A_23', 'xG_23', 'xAG_23', 'PrgC_23', 'PrgP_23', 'PrgR_23',
    'Min_24', '90s_24', 'G+A_24', 'xG_24', 'xAG_24', 'PrgC_24', 'PrgP_24', 'PrgR_24'
]

X = cleandf[features].fillna(0)
y = cleandf['Target_G+A'].fillna(0)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf_model = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
print(f"Model Accuracy (MAE): {np.round(mae):.3f} Goals/Assists off on average\n")

results = cleandf.loc[X_test.index, ['Player', 'Pos_23', 'Target_G+A']].copy()
results['Predicted_G+A'] = np.floor(y_pred)

top_predictions = results.sort_values(by='Predicted_G+A', ascending=False)

print("🏆 Model's Top 10 Predictions for the 25/26 Season (Test Set):")
print(top_predictions.head(10).to_string(index=False))

Model Accuracy (MAE): 2.000 Goals/Assists off on average

🏆 Model's Top 10 Predictions for the 25/26 Season (Test Set):
         Player Pos_23  Target_G+A  Predicted_G+A
Ousmane Dembélé  MF,FW          16           19.0
   Bryan Mbeumo     FW          12           18.0
Serhou Guirassy     FW          17           18.0
 Alexander Isak     FW           4           18.0
Bradley Barcola     FW          11           16.0
  Ollie Watkins     FW          13           14.0
Ademola Lookman  FW,MF           2           13.0
 Julián Álvarez  MF,FW          12           12.0
   Jacob Murphy  FW,DF           5           12.0
    Ayoze Pérez     FW           5           11.0


Final Predictions using the entire datset

In [64]:
all_predictions = rf_model.predict(X)

global_results = cleandf[['Player', 'Pos_23', 'Target_G+A']].copy()

global_results['Predicted_G+A'] = np.round(all_predictions).astype(int)

global_top = global_results.sort_values(by='Predicted_G+A', ascending=False)

print("🌍 GLOBAL TOP 10 PREDICTIONS (Entire Dataset):")
print(global_top.head(10).to_string(index=False))

🌍 GLOBAL TOP 10 PREDICTIONS (Entire Dataset):
         Player Pos_23  Target_G+A  Predicted_G+A
     Harry Kane     FW          38             29
  Michael Olise  FW,MF          33             25
 Erling Haaland     FW          32             25
  Kylian Mbappé     FW          28             24
   Lamine Yamal     FW          27             22
Bruno Fernandes  MF,FW          27             20
Mason Greenwood  MF,FW          21             19
Ousmane Dembélé  MF,FW          16             19
      Luis Díaz     FW          28             19
Serhou Guirassy     FW          17             19
